## **Building a Convolutional Neural Network (CNN) Architecture in PyTorch** 
___

This task translates a CNN architecture diagram into a working PyTorch `nn.Module` class. The diagram specifies a deep convolutional network: an input image passes through a convolution + pooling block, then three consecutive convolutions and a second pooling block, followed by dropout, flattening, and three fully connected layers ending in a softmax classifier.

The core challenge is calculating the spatial dimensions (height/width) at each stage, since the diagram left several of these blank (?) for the student to work out using the convolution/pooling output-size formula:

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Activity 6</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Convert the following CNN architecture diagram into a PyTorch CNN Architecture.
</p>

<center><img src="figures-quick_draw.png" width="400px"></center>
</div>

**Importing Required Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

**Setting the Input and Checking Dimensions** <br>

The CNN receives a batch of 32 grayscale images with a size of 28 × 28 pixels. A dummy input is created to trace how the dimensions change as the data passes through each layer. The output-size formula is also defined to verify the spatial dimensions of the convolution and pooling layers.

In [2]:
# ---------- Seed function----------
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

# ---------- Shape calculator ----------
def calc_out(w, f, s, p):
    print(f'Output Shape: {(w - f + 2 * p) // s + 1}')

set_seed(42)

# ---------- Dummy input to trace shapes (batch=32) ----------
images = torch.randn(32, 1, 28, 28)
print("Input shape:", images.shape)

Input shape: torch.Size([32, 1, 28, 28])


**First Convolution Layer** <br>

The first convolution changes the input from 1 channel to 32 feature maps. Because the layer uses a 3 × 3 kernel, stride 1, and padding 1, the spatial dimension remains 28 × 28. ReLU is then applied to introduce the activation function.

In [3]:
# Conv1 must output 28x28
calc_out(28, 3, 1, 1)

conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3,3), stride=1, padding=1)

x = F.relu(conv1(images))
print(x.shape)

Output Shape: 28
torch.Size([32, 32, 28, 28])


**First Max-Pooling Layer** <br>

The first max-pooling layer reduces the spatial dimensions from 28 × 28 to 15 × 15. The architecture specifies a 2 × 2 pooling window, stride 2, and padding 1, so the specified padding is retained when calculating the output size.

In [4]:
# Pool1 must output 15x15 (note: padding=1 here, unlike a typical pool)
calc_out(28, 2, 2, 1)

pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=1)

x = pool1(x)
print(x.shape)

Output Shape: 15
torch.Size([32, 32, 15, 15])


**Consecutive Convolution Layers** <br>

The next three convolutional layers progressively increase the number of feature maps from 32 to 64, then 128, and finally 256. Since all three layers use a 3 × 3 kernel, stride 1, and padding 1, the spatial dimensions remain at 15 × 15 while the number of extracted features increases.

In [5]:
# Each of these keeps spatial size the same (kernel=3, stride=1, padding=1 = "same" padding)
calc_out(15, 3, 1, 1)

conv2 = nn.Conv2d(32, 64, (3,3), 1, 1)
conv3 = nn.Conv2d(64, 128, (3,3), 1, 1)
conv4 = nn.Conv2d(128, 256, (3,3), 1, 1)

x = F.relu(conv2(x))
print(x.shape)

x = F.relu(conv3(x))
print(x.shape)

x = F.relu(conv4(x))
print(x.shape)

Output Shape: 15
torch.Size([32, 64, 15, 15])
torch.Size([32, 128, 15, 15])
torch.Size([32, 256, 15, 15])


**Second Max-Pooling Layer** <br>

The second max-pooling layer further reduces the spatial dimensions from 15 × 15 to 7 × 7. The number of feature maps remains 256 because pooling changes the spatial size but does not change the number of channels.

In [6]:
# Pool2 must output 7x7
calc_out(15, 2, 2, 0)

pool2 = nn.MaxPool2d((2,2), 2)

x = pool2(x)
print(x.shape)

Output Shape: 7
torch.Size([32, 256, 7, 7])


**Dropout and Flattening** <br>

After the second pooling layer, dropout with a probability of 0.2 is applied. The resulting 256 × 7 × 7 feature maps are then flattened into a one-dimensional representation. This produces 12,544 values per image, which becomes the input size of the first fully connected layer.

In [7]:
flatten_size = 256 * 7 * 7
print("Flatten size:", flatten_size)

dropout = nn.Dropout(p=0.2)
x = dropout(x)

flat = x.view(x.size(0), -1)
print("Flattened shape:", flat.shape)

Flatten size: 12544
Flattened shape: torch.Size([32, 12544])


**Fully Connected Layers** <br>

The flattened features are passed through three fully connected layers. The first layer reduces the 12,544 values to 1,000, followed by a second layer that reduces them to 500. The final layer produces 10 outputs, representing the classification results. ReLU is applied between the fully connected layers, followed by Softmax at the output.

In [17]:
fcn1 = nn.Linear(256*7*7, 1000)
fcn2 = nn.Linear(1000, 500)
fcn3 = nn.Linear(500, 10)  

out = F.relu(fcn1(flat))
out = F.relu(fcn2(out))
out = F.softmax(fcn3(out), dim=1)
print(out.shape)

torch.Size([32, 10])


**Combining the Layers into the CNN Architecture** <br>

After verifying the dimensions of each layer individually, the complete architecture is implemented as a PyTorch `nn.Module` class. The layers are defined in `__init__()` and connected sequentially in the `forward()` method to match the original CNN diagram.

In [13]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        # Conv1: 1 x 28 x 28 -> 32 x 28 x 28
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )
        
        # Pool1: 32 x 28 x 28 -> 32 x 15 x 15
        self.pool1 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=2,
            padding=1
        )

        # Conv2: 32 -> 64
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        # Conv3: 64 -> 128
        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        # Conv4: 128 -> 256
        self.conv4 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        # Pool2: 256 x 15 x 15 -> 256 x 7 x 7
        self.pool2 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=2,
            padding=0
        )

        # Dropout
        self.dropout = nn.Dropout(p=0.2)

        # Fully Connected Layers
        self.fcn1 = nn.Linear(256 * 7 * 7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)


    def forward(self, x):
        x = F.relu(self.conv1(x))   # Conv1 -> ReLU -> Pool1
        x = self.pool1(x)
        x = F.relu(self.conv2(x))   # Conv2 -> ReLU
        x = F.relu(self.conv3(x))   # Conv3 -> ReLU
        x = F.relu(self.conv4(x))   # Conv4 -> ReLU -> Pool2
        x = self.pool2(x)
        x = self.dropout(x)         # Dropout
        x = x.view(x.size(0), -1)   # Flatten
        x = F.relu(self.fcn1(x))    # FCN1 -> ReLU
        x = F.relu(self.fcn2(x))    # FCN2 -> ReLU
        x = F.softmax(self.fcn3(x), dim=1)  # FCN3 -> Softmax

        return x

**Inspecting the Completed Architecture** <br>

In [14]:
model = CNN(num_classes=10)   # replace 10 with your dataset's actual number of classes
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

>The completed CNN contains four convolutional layers, two max-pooling layers, dropout, and three fully connected layers. The architecture now follows the same sequence as the given diagram, with the calculated dimensions incorporated into the PyTorch implementation.

**Checking the Trainable Parameters**

In [15]:
def count_parameters(model):
    params = [p.numel() for p in model.parameters() if p.requires_grad]
    for item in params:
        print(f'{item:>8}')
    print(f'________\n{sum(params):>8}')

count_parameters(model)

     288
      32
   18432
      64
   73728
     128
  294912
     256
12544000
    1000
  500000
     500
    5000
      10
________
13438350


>The number of trainable parameters is counted to determine how many values the CNN needs to learn during training. The completed architecture contains 13,438,350 trainable parameters.

**Final Architecture Verification** <br>

The completed model is tested using a batch of 32 images with dimensions 1 × 28 × 28. The resulting output has a shape of 32 × 10, confirming that each input image passes successfully through the complete CNN and produces 10 classification outputs.

In [16]:
dummy_input = torch.randn(32, 1, 28, 28)

output = model(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])


The activity demonstrated how a CNN architecture diagram can be converted into a working PyTorch model by following the layers sequentially and checking the dimensions at each stage. The input started as a 1 × 28 × 28 grayscale image and passed through four convolutional layers, where the number of feature maps increased from 1 to 256. The two pooling layers reduced the spatial dimensions from 28 × 28 to 15 × 15 and finally to 7 × 7.

A key part of the process was determining the correct flattened size. After the second pooling layer, the feature maps had a size of 256 × 7 × 7, resulting in 12,544 values per image. This value was then used as the input size of the first fully connected layer. The final model produced 10 outputs for each input image, confirming that the architecture was successfully converted and connected according to the given diagram.

Overall, the process showed that checking the dimensions after each layer is important when implementing a CNN because each output becomes the input for the next layer. By verifying each stage first, the individual components could then be combined into a complete PyTorch CNN architecture.